# A free GPU for the song site

Open this in [Colab](https://colab.research.google.com/), choose **Runtime → Change
runtime type → T4 GPU**, and press **Run all**. Nothing is installed on your own
computer, nothing is rented, and no account or card is needed.

The last cell prints an `https://…trycloudflare.com` address. There are two ways to
use it:

1. **No key, no pool.** Open <https://browser.bpiersing.workers.dev>, press
   **Song designer**, open **Connect my model**, and paste that address. The page
   now writes lyrics and renders songs on this notebook's T4.
2. **Join the pool.** If the site owner has given you the join key, paste it when
   asked: this card is then added to the pool every other visitor gets, and no
   address has to be pasted anywhere. Press Enter to skip this.

The card goes away when the session ends — that is the trade for free. A T4 has
16 GB, twice the card the site was built on, so songs render faster while it runs.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())

In [ ]:
# The song service's own packages. Colab already ships a CUDA build of torch and
# torchaudio, so they are deliberately not named here: a pip wheel from PyPI would
# replace them with a CPU build and quietly move the model off the card.
!apt-get -qq install -y ffmpeg libsndfile1 git > /dev/null
!git clone --depth 1 https://github.com/soluzka/internet.git /content/internet 2>/dev/null || true
!pip install -q flask flask-cors soundfile librosa scipy demucs hf_transfer
!pip install -q transformers==4.57.6 huggingface-hub==0.36.2 numpy==2.4.6
!pip install -q git+https://github.com/ace-step/ACE-Step.git
import torch
print('packages in place; torch', torch.__version__, 'cuda', torch.cuda.is_available())

In [ ]:
# The two checkpoints (about 10 GB, into the session disk): ACE-Step v1 3.5B for
# the music, Qwen2.5 1.5B for the words. install.py reads song_service/models.json
# and resumes rather than restarting a partial folder.
import os, subprocess
os.environ['HF_HOME'] = '/content/hf'
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
result = subprocess.run(['python', 'song_service/install.py', '--weights'], cwd='/content/internet')
print('weights exit', result.returncode)
!du -sh /content/internet/song_service/models/* 2>/dev/null || true

In [ ]:
# The service on the notebook's own GPU, on every interface, warming the model now
# so the first visitor does not wait for the load.
import os, subprocess, time, urllib.request
os.environ.update(SONG_PORT='8090', SONG_HOST='0.0.0.0', SONG_WARM='1', HF_HOME='/content/hf')
log = open('/content/song.log', 'w')
proc = subprocess.Popen(['python', 'app.py', '--role', 'song'], cwd='/content/internet/song_service',
                        stdout=log, stderr=subprocess.STDOUT)
health = ''
for _ in range(90):
    try:
        health = urllib.request.urlopen('http://127.0.0.1:8090/health', timeout=5).read().decode()
        break
    except Exception:
        time.sleep(5)
print('service:', health[:300] or 'not answering yet, see /content/song.log')

In [ ]:
# A free tunnel, so the Cloudflare page can reach this card. Nothing is opened on
# the notebook host: cloudflared dials out, exactly like the launcher does at home.
import re, time, json, subprocess, urllib.request, getpass
!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /usr/local/bin/cloudflared

tunnel = subprocess.Popen(['cloudflared', 'tunnel', '--no-autoupdate', '--url', 'http://127.0.0.1:8090',
                           '--logfile', '/content/tunnel.log'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
url = ''
for _ in range(40):
    try:
        text = open('/content/tunnel.log').read()
        found = re.findall(r'https://[a-z0-9-]+[.]trycloudflare[.]com', text)
        if found:
            url = found[-1]
            break
    except FileNotFoundError:
        pass
    time.sleep(2)

print()
print('=' * 70)
print('tunnel:', url or 'not up yet, see /content/tunnel.log')
print('=' * 70)
print()
if url:
    print('Option 1 - no key needed:')
    print('  open https://browser.bpiersing.workers.dev')
    print('  Song designer -> Connect my model -> paste this address:')
    print('   ', url)
    print()
    print('Option 2 - add this card to the pool for every visitor (needs the key):')
    key = getpass.getpass('Join key from the site owner (press Enter to skip): ')
    if key:
        body = json.dumps({'url': url, 'note': 'colab t4'}).encode()
        request = urllib.request.Request('https://song-pool.bpiersing.workers.dev/join', data=body,
                                         headers={'content-type': 'application/json', 'x-join-key': key})
        try:
            print(urllib.request.urlopen(request, timeout=20).read().decode())
            print('This card is in the pool until the notebook stops. Keep the tab open.')
        except Exception as exc:
            print('join refused:', exc)
            print('Use option 1 with the address above instead.')
    else:
        print('Skipped. Use option 1 with the address above - this card is reachable,')
        print('it is just not in the pool.')
else:
    print('No tunnel address yet: rerun this cell, or check /content/tunnel.log.')